# GeoBrix Sample Data Setup

Run this notebook on a **Databricks cluster** to download the Essential and Complete sample-data bundles into a Unity Catalog Volume. Uses the GeoBrix **`sample`** module (included in the GeoBrix WHL)—install GeoBrix and run the cells in order.

- Skips datasets that already exist at the Volumes path
- Uses a local temp directory for interim downloads (zip, convert); only final files are copied to the Volume

## 1. Config: catalog, schema, volume

In [ ]:
from databricks.labs.gbx.sample import get_volumes_path, run_essential_bundle, run_complete_bundle
from pathlib import Path

CATALOG = "main"
SCHEMA = "default"
VOLUME = "geobrix_samples"
SAMPLE_DATA_PATH = get_volumes_path(CATALOG, SCHEMA, VOLUME)
print(f"SAMPLE_DATA_PATH = {SAMPLE_DATA_PATH}")

## 2. Install dependencies (once per session)

In [ ]:
%pip install -q requests pystac-client planetary-computer geopandas

## 3. Run Essential Bundle (~355 MB)

NYC & London vectors (GeoJSON), Sentinel-2 rasters, SRTM elevation. Skips files that already exist.

In [ ]:
Path(SAMPLE_DATA_PATH).mkdir(parents=True, exist_ok=True)
result_essential = run_essential_bundle(SAMPLE_DATA_PATH)
print(f"Files: {result_essential['file_count']}, Total: {result_essential['total_size_mb']:.1f} MB")
if result_essential["errors"]:
    for name, err in result_essential["errors"]:
        print(f"  ⚠️ {name}: {err[:80]}...")

## 4. Run Complete Bundle (~440 MB more)

Neighborhoods, extra elevation, HRRR weather, shapefiles (parks, subway), multi-layer GeoPackage. Run after Essential.

In [ ]:
result_complete = run_complete_bundle(SAMPLE_DATA_PATH)
print(f"Files: {result_complete['file_count']}, Total: {result_complete['total_size_mb']:.1f} MB")
if result_complete["errors"]:
    for name, err in result_complete["errors"]:
        print(f"  ⚠️ {name}: {err[:80]}...")

## 5. Summary

In [ ]:
print(f"Sample data base path: {SAMPLE_DATA_PATH}")
for item in sorted(Path(SAMPLE_DATA_PATH).rglob("*")):
    if item.is_file():
        rel = item.relative_to(SAMPLE_DATA_PATH)
        print(f"  {rel}: {item.stat().st_size / (1024*1024):.1f} MB")